In [ ]:
*Title:* 7_Permutations
*Goal:* Generate permutations for each repeatability type

Based on permutations.sh

# 7.1_Generate_Datasets 
*Goal:* 
- Generate 1000 datasets with random 5% of SNPs in each population as outliers, and then generating counts of each type in all permutations. Then compare the counts observed with the permutation results.
 with the outlier results: 4.outliers_poolfstat_SNP_FST_95_2016_2017.tsv

Confirm Number of SNPs in real data:
```bash
awk -F'\t' '{print NF; exit}' 4.outliers_poolfstat_SNP_FST_95_2016_2017.tsv
=6261839 -1 = 6261838 
```

Get sample names in correct order:
```bash
cut -f1 4.outliers_poolfstat_SNP_FST_95_2016_2017_outliers_only_AGAIN.tsv | nano -
#saved file as population.tsv
```
Calculate permutations*

```r
library(data.table)
library(tidyverse)

# Define the sample names
num_snps <- 6261838   # Number of SNPs (columns)
num_permutations <- 1000  # Number of permutations
percent_true <- 0.05  # Percentage of "TRUE" values per sample
random_snps <- data.frame(SNP_ID = 1:num_snps)

sample_names <- c("Younger2016","OldDairy2016","Scott2016","Lombardi2016","Laguna2016","Waddell2016","Younger2017",
"OldDairy2017","Scott2017","Lombardi2017","Laguna2017","Waddell2017")

# Update the number of samples
num_samples <- length(sample_names)

# Perform permutations and save each individually
set.seed(123)  # For reproducibility
for (perm in 1:num_permutations) {
  random_snps <- data.frame(SNP_ID = 1:num_snps) # Create SNP dataframe
  
  # Generate random 5% SNPs for each sample
  for (sample in 1:num_samples) {
    # Randomly select 5% of SNPs without replacement
    random_indices <- sample(1:num_snps, size = ceiling(percent_true * num_snps), replace = FALSE)
    
    # Create a column for the sample and mark selected SNPs as TRUE
    random_snps[[sample_names[sample]]] <- FALSE
    random_snps[random_indices, sample_names[sample]] <- TRUE
  }

   reshaped_data <- random_snps %>% #reshaping to match what I did before.
    pivot_longer(cols = -SNP_ID, names_to = "Population", values_to = "Value") %>%
    pivot_wider(names_from = SNP_ID, values_from = Value)
  
  
  # Save the result for this permutation to a file
  output_file <- paste0("random_snps_permutation_", perm, ".tsv")
  write.table(reshaped_data, file = output_file, sep = "\t", row.names = FALSE, quote = FALSE)
  print(paste(output_file,"done"))
}
```
Now gets counts
*Goal:* Get the occurences of each repeatability type in each permutation.
```bash
#Counts:
for input_file in random_snps_permutation_*.tsv; do
    # Extract the permutation number from the file name
    output_file="repeatability_${input_file%.tsv}.tsv"

awk 'BEGIN {
    FS = OFS = "\t" # Set field separator (FS) and output field separator (OFS) to tab
    comb[1] = "2 8" # Define row combinations
    comb[2] = "3 9"
    comb[3] = "4 10"
    comb[4] = "5 11"
    comb[5] = "6 12"
    comb[6] = "7 13"
    comb[7] = "2 3 4 5 6 7"
    comb[8] = "8 9 10 11 12 13"
    names[1] = "tYounger" # Define category names
    names[2] = "tOldDairy"
    names[3] = "tScott"
    names[4] = "tLombardi"
    names[5] = "tLaguna"
    names[6] = "tWaddell"
    names[7] = "spatial_2016"
    names[8] = "spatial_2017"
}
NR == 1 {
    split($0, headers, FS) # Split the header line into an array
    next
}
{
    for (i = 1; i <= 8; i++) { # Loop over combinations
        split(comb[i], rows, " ") # Split the combination into row indices
        for (j = 1; j <= NF; j++) { # Loop over fields in the current line
            for (k in rows) { # Loop over row indices
                if (NR == rows[k] && $j == "TRUE") { # Check row and value
                    sum[i, j]++ # Increment count for this combination and field
                }
            }
        }
    }
    for (j = 1; j <= NF; j++) { # Loop over fields in the current line
        total[j] += ($j == "TRUE") # Increment total count for each field if value is "TRUE"
    }
}
END {
    print "ColumnNumber", "Count", "tYounger", "tOldDairy", "tScott", "tLombardi", "tLaguna", "tWaddell", "spatial_2016", "spatial_2017"
    for (j = 1; j <= NF; j++) {
        printf "%s\t%d", headers[j], total[j] # Print the header instead of column number
        for (i = 1; i <= 8; i++) printf "\t%d", sum[i, j] + 0
        print ""
    }
}' "$input_file" > "$output_file"
echo "$output_file done"
done
```
Clean the file!
```r
    #This output file will contain the number of times a SNP is an outlier in each estuary, and in each year. 
    #The output file will have the following columns:
        #ColumnNumber: The column number in the input file
        #Count: The total number of times the SNP is an outlier
        #tYounger: The number of times the SNP is an outlier in the Younger estuary
        #tOldDairy: The number of times the SNP is an outlier in the Old Dairy estuary
        #tScott: The number of times the SNP is an outlier in the Scott estuary
        #tLombardi: The number of times the SNP is an outlier in the Lombardi estuary
        #tLaguna: The number of times the SNP is an outlier in the Laguna estuary
        #tWaddell: The number of times the SNP is an outlier in the Waddell estuary
        #spatial-2016: The number of times the SNP is an outlier in 2016
        #spatial-2017: The number of times the SNP is an outlier in 2017
        #spatial_temporal_count= Means the number of times (estuaries) that a given SNP is an outlier in both years.
        #only_2016_spatial: The number of times the SNP is an outlier in 2016, but not in 2017
        #only_2017_spatial: The number of times the SNP is an outlier in 2017, but not in 2016
#here, adding columns for pure spatial counts (i.e. counts if only an outlier in 2016 and only an outlier in 2017)
    #This gives me the pure outliers (not temporally repeated...)
#also dividing the counts by 2 and removing the rows where the count is 1.
    #This is because the counts are doubled in the upset script, and we only want to keep the SNPs that are outliers in both years.
    #i.e. I want a tLBD to mean that it is a temporal outlier in LBD, that that it is an outlier for LBD in either 2016 or 2017. Currently, the script is just counting T for LBD 2016 and 2017, so a count of 2 actually means temporally repeated. Below script fixes that!
library(data.table)
library(tidyverse)

file_list<-list.files(pattern="repeatability_random_snps_permutation_.*\\.tsv$")

for (file in file_list) {
data<- read.table(file, header=TRUE, sep="\t")
data<-data[-1,] #removing extra row
data$only_2016_spatial<- ifelse (data$spatial_2017 == 0, data$spatial_2016, 0) #making a column for pure spatial 2016 (not in 2017 at all). If not pure to 2016, a 0 is printe.d
data$only_2017_spatial<- ifelse (data$spatial_2016 == 0, data$spatial_2017, 0)  #making a column for pure spatial 2017 (not in 2016 at all)
columns_to_modify <- c("tYounger", "tOldDairy", "tScott", "tLombardi", "tLaguna", "tWaddell")
data[columns_to_modify] <- lapply(data[columns_to_modify], function(x) x/2)
data[columns_to_modify] <- lapply(data[columns_to_modify], function(x) ifelse(x == 0.5, 0, x)) #removing partial temporals
data$spatial_temporal_count <- rowSums(data[columns_to_modify])

#Make local counts:
data$Local_Younger <- ifelse(data$tYounger == 1 & data$tOldDairy == 0 & data$tWaddell == 0
& data$tLaguna == 0 & data$tLombardi == 0 & data$tScott == 0, 1, 0)

data$Local_OldDairy <- ifelse(data$tOldDairy == 1 & data$tYounger == 0 & data$tWaddell == 0
& data$tLaguna == 0 & data$tLombardi == 0 & data$tScott == 0, 1, 0)

data$Local_Waddell <- ifelse(data$tWaddell == 1 & data$tOldDairy == 0 & data$tYounger == 0
& data$tLaguna == 0 & data$tLombardi == 0 & data$tScott == 0, 1, 0)

data$Local_Laguna <- ifelse(data$tLaguna == 1 & data$tOldDairy == 0 & data$tWaddell == 0
& data$tYounger == 0 & data$tLombardi == 0 & data$tScott == 0, 1, 0)

data$Local_Lombardi <- ifelse(data$tLombardi == 1 & data$tOldDairy == 0 & data$tWaddell == 0
& data$tLaguna == 0 & data$tYounger == 0 & data$tScott == 0, 1, 0)

data$Local_Scott <- ifelse(data$tScott == 1 & data$tOldDairy == 0 & data$tWaddell == 0
& data$tLaguna == 0 & data$tLombardi == 0 & data$tYounger == 0, 1, 0)

output_file <- paste0(tools::file_path_sans_ext(basename(file)), "_cleaned.tsv")
#save output as .tsv file:
write.table(data, file= output_file, sep="\t", row.names=FALSE, quote=FALSE)

#Now gathering results for summary counts/permutation dataset:
#Gathering and saving results:
spatial_temporal_counts<- as.data.frame(table(data$spatial_temporal_count))
only_2016_spatial_counts<- as.data.frame(table(data$only_2016_spatial))
only_2017_spatial_counts<- as.data.frame(table(data$only_2017_spatial))
spatial_2016_counts<- as.data.frame(table(data$spatial_2016))
spatial_2017_counts<-as.data.frame(table(data$spatial_2017))

#Changing the column maes for later merging:
colnames(spatial_temporal_counts) <- c("Number_of_estuaries", "spatial_temporal_counts")
colnames(only_2016_spatial_counts) <- c("Number_of_estuaries", "only_2016_spatial_counts")
colnames(only_2017_spatial_counts) <- c("Number_of_estuaries", "only_2017_spatial_counts")
colnames(spatial_2016_counts) <- c("Number_of_estuaries", "spatial_2016_counts")
colnames(spatial_2017_counts) <- c("Number_of_estuaries", "spatial_2017_counts")


summary_repeatabililty_counts <- Reduce(function(x, y) merge(x, y, by = "Number_of_estuaries", all = TRUE), 
                                        list(spatial_temporal_counts, only_2016_spatial_counts, 
                                             only_2017_spatial_counts, spatial_2016_counts, 
                                             spatial_2017_counts))

#Replace NA with 0:
summary_repeatabililty_counts[is.na(summary_repeatabililty_counts)] <- 0
#Save as .tsv file:
output_file <- paste0(tools::file_path_sans_ext(basename(file)), "_repeatability_summary.tsv")
write.table(summary_repeatabililty_counts, file=output_file, sep="\t", row.names=FALSE, quote=FALSE)
print(paste(output_file,"done"))
}
```

# 7.2_P_Values
*Goal:* Compare permutation output to observed, to calculate p-values.

```r
#Stuff here based on what I did in Graphs for Repeatability Counts (to observed data)
library(data.table)
library(tidyverse)
library(xtable)
library(tinytex)

#First loading all permutation files 
file_list <- list.files(pattern = "._repeatability_summary\\.tsv$")
data_list <- list()

for (file in file_list) {
  data <- read.table(file, header = TRUE, sep = "\t", row.names=NULL)
  data$file_name <- file  # Add the file name as a new column
  data_list[[file]] <- data  # Store the data frame in the list
}

combined_data <- do.call(rbind, data_list)
combined_data$perm_number<-str_extract(combined_data$file_name, "\\d+")

permutation_data<-combined_data[,c(1:4,8)] 

observed_data <- read.table("summary_repeatabililty_counts_AGAIN.tsv", header=TRUE, sep="\t")

observed_data<-observed_data[,c(1:4)] %>%
mutate(perm_number = "observed")

combined_data_all <- bind_rows(permutation_data, observed_data) #merging observed and permutation data

#Now I want to clean up hte dataset, by removing the 0 estuaries, and renaming. I will group the spatials together by summing them (i.e. spatial=spatial 2016+2017), representing spatial repeatability in a given estuary/sharing for either year
combined_data_all$Number_of_estuaries <- as.numeric(combined_data_all$Number_of_estuaries)

combined_data_all<-combined_data_all %>%
filter(Number_of_estuaries>0) #removing 0 estuaries.

combined_data_all_long <- pivot_longer(combined_data_all, 
                                cols =c(-Number_of_estuaries, -perm_number), #keep these columns as they are
                                names_to = "Type", 
                                values_to = "value") #reformatiing for graphing

#now renaming and summing columns:
combined_data_all_long <- combined_data_all_long %>%
  mutate(Type = ifelse(Number_of_estuaries == 1 & Type=="spatial_temporal_counts","Temporal", Type))

#Renaming the types:
combined_data_all_long <- combined_data_all_long %>%
  mutate(Type = recode(Type,
                       "spatial_temporal_counts" = "Spatio-temporal",
                       "only_2016_spatial_counts" = "Spatial_2016",
                       "only_2017_spatial_counts" = "Spatial_2017"))

#Combining the years into spatial (either 2016 or 2017):
#I want to make a new category called spatial which is the sum of spatial 2016 and 2017 for the same estuary count:
combined_data_all_long <- combined_data_all_long %>%
  mutate(Type = ifelse(Type == "Spatial_2016" | Type == "Spatial_2017", "Spatial", Type))
#Now I want to group by the number of estuaries and type, and sum the values:
combined_data_all_long <- combined_data_all_long %>%
  group_by(Number_of_estuaries, Type, perm_number) %>%
  summarise(value = sum(value)) %>%
  ungroup()

#Here, I want to make sure that estuaries 1:6 are represented for all types/permutations, so things don't get funky later
combined_data_all_long <- combined_data_all_long %>%
  complete(
    Number_of_estuaries = 1:6,  # Ensure all estuaries 1 through 6 are included
    Type,
    perm_number,
    fill = list(value = 0)  # Fill missing values with 0
  )


#Now split the datasets, so a dataset per number of estuaries/Type:
split_datasets <- split(combined_data_all_long, 
                        list(combined_data_all_long$Number_of_estuaries, combined_data_all_long$Type)) #this splits the data into separate lists for later stuff


results_list <- list()

# Iterate through each split dataset
for (name in names(split_datasets)) {
  # Get the current dataset
  dataset <- split_datasets[[name]]
  
  # Extract observed values and permutation values
  observed_value <- dataset %>% filter(perm_number == "observed") %>% pull(value)
  permutation_values <- dataset %>% filter(perm_number != "observed") %>% pull(value)
  
    if (length(observed_value) == 0 || length(permutation_values) == 0) {
    results_list[[name]] <- data.frame(
      Dataset = name,
      Observed = NA,
      Greater_Than = NA,
      P_Value = NA
    )
    next
  }

  # Calculate p-value
  greater_than<-sum(permutation_values >= observed_value)
  p_value <-  (greater_than+1)/(1000 +1) #Here I am writing 1000, because there were 1000 permutations. Not the same thing as length, because not all of the permutations had values for each type/nuber of estuary (i.e. not all had values at 5 or 6 estuaries). Added a +1 correction based on Phipson and Smyth (2010) 
  
  results_list[[name]] <- data.frame(
    Dataset = name,
    Observed = observed_value,
    Greater_Than = greater_than,
    P_Value = p_value
  )

}

#Now I want to save the data, showing the number observed, the number greater than or equal (of permutations), p-value, and the expected (based on probabilities):
results_table <- do.call(rbind, results_list)
results_table<-na.omit(results_table)
results_table$Type<-str_extract(results_table$Dataset,"Spatial|Temporal|Spatio-temporal")
results_table$Number_of_estuaries<-str_extract(results_table$Dataset,"\\d+")

#Calculate the Expected Occurences (based on random chance):
n <- 6  # Number of estuaries in 2016
m <- 6  # Number of estuaries in 2017
p <- 0.05  # Probability of being an outlier in a single estuary (T)
p_squared <- p^2  # Probability of being an outlier in both years
total_snps <- 6261838
calculate_probability <- function(k, n, p_squared) {
  binomial_coefficient <- choose(n, k)  # Calculate binomial coefficient
  prob <- binomial_coefficient * (p_squared^k) * ((1 - p_squared)^(n - k))  # Formula
  return(prob)
} #p_squared^k refers to the chance of getting t/t, 1 - p_squared ensures that all other estuaries AREN"T t/t (can be t/f or f/t or f/f)

spatio_temporal_probabilities <- sapply(2:6, function(k) calculate_probability(k, n, p_squared)) #Where k=1 would be temporal and all others would be spatio-temporal.


# Spatial Repeatability
calculate_spatial <- function(k, n, m, p) {
  binomial_coefficient <- choose(n, k)
  prob_2016 <- binomial_coefficient * (p^k) * ((1 - p)^(n - k)) #probability of spatial repeatability in 2016
  prob_2017 <- (1 - p)^m #probability that it will not be an outlier in ANY estuary in 2017
  overall_probability <- prob_2016 * prob_2017
  return(overall_probability * 2)  # Multiply by 2 for 2016 or 2017
}
# Calculate probabilities for spatial repeatability (k = 2 to 6)
spatial_probabilities <- sapply(1:6, function(k) calculate_spatial(k, n, m, p))


# Temporal Repeatability
prob_shared_temp <- n * p_squared * ((1 - p^2)^5)

results_df <- data.frame(
  Number_of_estuaries = 1:6,
  Spatial_Probability = c(spatial_probabilities),  # NA for k = 1
  Spatio_Temporal_Probability = c(NA,spatio_temporal_probabilities),
  Temporal_Probability = c(prob_shared_temp, rep(NA, 5))  # Only for k = 1
)


# Expected number of SNPs for each category
expected_spatial <- spatial_probabilities * total_snps
expected_spatio_temporal <- spatio_temporal_probabilities * total_snps
expected_temporal <- prob_shared_temp * total_snps

# Combine expected counts into a dataframe
expected_df <- data.frame(
  Number_of_estuaries = 1:6,
  Expected_Spatial = c(expected_spatial),  # NA for k = 1
  Expected_Spatio_Temporal = c(NA,expected_spatio_temporal),
  Expected_Temporal = c(expected_temporal, rep(NA, 5))  # Only for k = 1
)
#Reorganize pivot_longer by type:
expected_df <- expected_df %>%
  pivot_longer(cols = -Number_of_estuaries, 
               names_to = "Type", 
               values_to = "Expected_Counts") %>%
  mutate(Type = recode(Type,
                       Expected_Spatial = "Spatial",
                       Expected_Spatio_Temporal = "Spatio-temporal",
                       Expected_Temporal = "Temporal"))

results_table$Number_of_estuaries<-as.numeric(results_table$Number_of_estuaries)
results_table$Observed<-as.numeric(results_table$Observed)
#Now combine everything:
results_table <- results_table %>%
  left_join(expected_df, by = c("Number_of_estuaries", "Type")) 

results_table <- results_table %>%
  mutate(Observed_Expected_Ratio = Observed/Expected_Counts)

#save results:
write.table(results_table, "permutation_observed_expected_Results_Repeatability_type.tsv", sep = "\t", row.names = FALSE, quote = FALSE)


#Now making a table:
data <-results_table[,c(1:6,8)] %>%
mutate(Type_Estuaries = paste(Type, Number_of_estuaries, sep =" "))%>%
select(c("Type_Estuaries","Observed","Greater_Than","P_Value","Observed_Expected_Ratio"))%>%
rename(
  'Repeatability Type' = Type_Estuaries,
  'Ratio'=Observed_Expected_Ratio,
  'Permutations >= Observed'= Greater_Than,
  'p'= P_Value) %>%
  na.omit(data) 
  
  data_final<- 
  data[2:12,]%>%
   mutate(`Repeatability Type` = ifelse(`Repeatability Type` == "Temporal 1", "Temporal", `Repeatability Type`))%>%
   rename("p-value permutations"="p")%>%
   select("Repeatability Type", "Observed","Ratio","p-value permutations")

#adjusting the sig figs shown per column: 
data_final$Observed <- sprintf("%.0f", data_final$Observed)
data_final$`p-value permutations` <- sprintf("%.3f", as.numeric(data_final$`p-value permutations`))

# Create xtable with the formatted data  
xtable_df <- xtable(data_final, 
                    caption = "Repeatability Frequencies",
                    align = rep("c", ncol(data_final) + 1))

# Save just the table content
print(xtable_df, 
      type = "latex",
      file = "table_content.tex", 
      include.rownames = FALSE,
      floating = FALSE)

# Create the main LaTeX document
latex_document <- "\\documentclass{article}
\\usepackage{booktabs}
\\usepackage{longtable}
\\usepackage{amssymb}
\\usepackage{geometry}
\\usepackage{rotating}
\\geometry{a4paper, margin=1in}

\\begin{document}
\\begin{sidewaystable}
\\resizebox{\\textwidth}{!}{
\\input{table_content.tex}
}
\\end{sidewaystable}
\\end{document}"

# Write the main LaTeX document
writeLines(latex_document, "permutation_observed_expected_Results_Repeatability_type.tex")

# Compile to PDF
tinytex::latexmk("permutation_observed_expected_Results_Repeatability_type.tex")